# Lab 1: Simple Object Detection and Benchmarking with OpenVINO

**Goals:**
- Run object detection using OpenVINO
- Understand inference flow
- Measure throughput and FPS

## Installation

Run the cell below once to install required libraries.

In [ ]:
# Install required libraries (run once)
!pip install openvino opencv-python numpy matplotlib ipywidgets

## Simple Pipeline

**Input Image** → **Preprocessing** → **OpenVINO Model** → **Detection Output** → **Bounding Boxes**

## Setup

Run this notebook from the `1/` directory. Copy both this notebook and `utils.py` to your server—they must be in the same folder.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import openvino as ov
import os
from ipywidgets import Dropdown
from IPython.display import display

from utils import find_model_xml, preprocess_image, run_inference, postprocess, draw_boxes, benchmark_latency_ms

# Configuration
IMAGE_PATH = "media/sample_image-1.jpg"
PRECISION = "FP16"

model_dropdown = Dropdown(options=["ATSS-MobileNetV2", "Deim-DFine-X"], value="ATSS-MobileNetV2", description="Model:")
device_dropdown = Dropdown(options=["CPU", "GPU", "NPU"], value="CPU", description="Device:")
display(model_dropdown, device_dropdown)

## Model Loading

In [ ]:
MODEL_NAME = model_dropdown.value
model_dir = os.path.join("models", MODEL_NAME, PRECISION)
xml_path = find_model_xml(model_dir)

core = ov.Core()
model = core.read_model(xml_path)
compiled_model = core.compile_model(model, device_dropdown.value)

input_layer = compiled_model.input(0)
input_shape = input_layer.shape
print(f"Model loaded: {xml_path}")
print(f"Device: {device_dropdown.value}")
print(f"Input shape: {input_shape}")

## Image Inference

In [ ]:
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

orig_h, orig_w = image.shape[:2]
input_tensor = preprocess_image(image, input_shape)
output = run_inference(compiled_model, input_tensor)

# DEBUG: Remove once detection works. Share output if 0 detections.
print("=== Model output (remove when working) ===")
for k, v in output.items():
    a = np.array(v)
    flat = a.flatten()
    print(f"  {k}: shape={a.shape}, dtype={a.dtype}")
    print(f"    first 14: {flat[:14].tolist()}")
    if a.size > 0:
        print(f"    min={float(np.min(a)):.4f}, max={float(np.max(a)):.4f}")
    if k.lower() == "boxes" and a.ndim >= 2 and a.shape[-1] >= 5:
        arr = a.reshape(-1, a.shape[-1])
        print(f"    row0 col4 (conf?): {float(arr[0, 4]) if len(arr) > 0 else 'N/A'}")
print("==========================================")

_, _, input_h, input_w = input_shape
boxes = postprocess(output, orig_h, orig_w, input_h, input_w, MODEL_NAME)
result_img = draw_boxes(image, boxes)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Detections: {len(boxes)} objects")
plt.show()

## Optional: PyTorch to OpenVINO Conversion

For reference only. The main workflow uses pre-converted IR models. Use `convert_model()` and `ov.save_model()` to export PyTorch models to OpenVINO IR.

## Benchmarking: Inference Time by Precision

Single-image object detection. Average inference time (ms) for FP32, FP16, and INT8.

In [ ]:
results = {}
for prec in ["FP32", "FP16", "INT8"]:
    model_dir_p = os.path.join("models", MODEL_NAME, prec)
    try:
        xml_path_p = find_model_xml(model_dir_p)
        model_p = core.read_model(xml_path_p)
        compiled_p = core.compile_model(model_p, device_dropdown.value)
        lat_ms = benchmark_latency_ms(compiled_p, input_tensor)
        results[prec] = lat_ms
        print(f"{prec} → {lat_ms:.1f} ms")
    except FileNotFoundError:
        print(f"{prec} → (model not found)")
        results[prec] = 0
    except Exception as e:
        print(f"{prec} → Error: {e}")
        results[prec] = 0

precisions = [p for p in results if results[p] > 0]
vals = [results[p] for p in precisions]
if precisions:
    plt.figure(figsize=(6, 4))
    plt.bar(precisions, vals, color=["#2ecc71", "#3498db", "#e74c3c"], edgecolor="black")
    plt.xlabel("Precision")
    plt.ylabel("Average Inference Time (ms)")
    plt.title("Average Inference Time by Precision")
    plt.tight_layout()
    plt.show()

## Wrap-up

- OpenVINO simplifies deployment of object detection models
- The detection pipeline is straightforward: load → preprocess → infer → postprocess
- Lower precision (FP16, INT8) typically reduces inference time
- Pre-converted IR models keep the workflow simple and fast